In [ ]:
!pip install transformers datasets scikit-learn streamlit pyngrok -q

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from transformers import (
    BertTokenizer,
    BertForSequenceClassification,
    TrainingArguments,
    Trainer
)
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
dataset = load_dataset("ag_news")
label_names = dataset["train"].features["label"].names
print("Labels:", label_names)
print("Sample:", dataset["train"][0])

In [ ]:
tokenizer = BertTokenizer.from_pretrained("bert-base-uncased")

def tokenize(batch):
    return tokenizer(
        batch["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )

tokenized = dataset.map(tokenize, batched=True)
tokenized.set_format("torch", columns=["input_ids", "attention_mask", "label"])

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, predictions)
    f1  = f1_score(labels, predictions, average="weighted")
    return {"accuracy": acc, "f1": f1}

In [ ]:
training_args = TrainingArguments(
    output_dir="./bert-ag-news",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=100,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    report_to="none"
)

In [ ]:
# Balanced sample: 2500 per class = 10,000 total (manageable + accurate)
def balanced_sample(dataset, per_class=2500):
    indices = []
    for label in range(4):
        class_indices = [i for i, x in enumerate(dataset) if x["label"] == label]
        indices += class_indices[:per_class]
    return dataset.select(indices)

train_dataset = balanced_sample(tokenized["train"], per_class=2500)
eval_dataset  = tokenized["test"].select(range(2000))

print(f"Train size: {len(train_dataset)}")
print(f"Eval size:  {len(eval_dataset)}")

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    compute_metrics=compute_metrics
)

trainer.train()

In [ ]:
results = trainer.evaluate()
print(f"\n✅ Accuracy : {results['eval_accuracy']:.4f}")
print(f"✅ F1 Score : {results['eval_f1']:.4f}")

In [ ]:
trainer.save_model("./bert-ag-news")
tokenizer.save_pretrained("./bert-ag-news")
print("✅ Model saved!")

In [ ]:
app_code = '''
import streamlit as st
import torch
from transformers import BertTokenizer, BertForSequenceClassification

@st.cache_resource
def load_model():
    tokenizer = BertTokenizer.from_pretrained("./bert-ag-news")
    model = BertForSequenceClassification.from_pretrained("./bert-ag-news")
    model.eval()
    return tokenizer, model

label_names = ["World", "Sports", "Business", "Sci/Tech"]
label_icons = ["🌍", "⚽", "💼", "🔬"]
label_colors = ["#4A90D9", "#27AE60", "#E67E22", "#8E44AD"]

st.set_page_config(page_title="News Classifier", page_icon="📰", layout="centered")

st.markdown("""
    <style>
        .main { background-color: #f9f9f9; }
        .result-box {
            padding: 1.2rem 1.5rem;
            border-radius: 12px;
            background: #ffffff;
            border-left: 6px solid;
            box-shadow: 0 2px 10px rgba(0,0,0,0.08);
            margin-top: 1rem;
        }
    </style>
""", unsafe_allow_html=True)

st.markdown("# 📰 News Topic Classifier")
st.markdown("Fine-tuned BERT on AG News — World · Sports · Business · Sci/Tech")
st.divider()

tokenizer, model = load_model()

headline = st.text_input("Enter a news headline:", placeholder="e.g. Tesla stock surges after record earnings")

if st.button("🔍 Classify", use_container_width=True):
    if not headline.strip():
        st.warning("Please enter a headline first.")
    else:
        with st.spinner("Analyzing..."):
            inputs = tokenizer(
                headline,
                return_tensors="pt",
                truncation=True,
                padding=True,
                max_length=128
            )
            with torch.no_grad():
                outputs = model(**inputs)

            probs = torch.softmax(outputs.logits, dim=-1)[0]
            pred_id = torch.argmax(probs).item()
            confidence = probs[pred_id].item()

        # ✅ Use st. functions directly — no f-string HTML needed
        st.success(f"{label_icons[pred_id]}  {label_names[pred_id]}  —  {confidence*100:.1f}% confidence")

        st.markdown("#### All Category Scores")
        for i in range(4):
            st.progress(
                float(probs[i]),
                text=f"{label_icons[i]} {label_names[i]}: {probs[i]*100:.1f}%"
            )

st.divider()
st.caption("Model: bert-base-uncased | Dataset: AG News | Balanced 10k training samples")
'''

with open("app.py", "w") as f:
    f.write(app_code)

print("✅ app.py written!")

In [ ]:
!pip install pyngrok -q

from pyngrok import ngrok
import subprocess, time

# ✅ Paste your token from https://dashboard.ngrok.com
ngrok.set_auth_token("ENTER YOUR AUTH TOKEN HERE")

# Kill old processes
!pkill -f streamlit 2>/dev/null
!pkill -f ngrok    2>/dev/null
time.sleep(2)

# Start streamlit
subprocess.Popen([
    "streamlit", "run", "app.py",
    "--server.port=8501",
    "--server.headless=true",
    "--server.enableCORS=false",
    "--server.enableXsrfProtection=false"
], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

time.sleep(5)

# Open tunnel
url = ngrok.connect(8501, bind_tls=True)
print(f"\n✅ App is live at: {url.public_url}")
print("👆 Click the link above to open your app!")